# Visualização Interativa com Plotly, Bokeh e Altair

Gráficos interativos, hover, exportação e uma comparação declarativa com Altair. Dados sintéticos; mapas demonstrativos não representam medidas reais.

**Objetivo:** escolher conscientemente entre exploração rápida, controle fino com callbacks e gramática declarativa de visualizações.

### Materiais complementares

- [Guia — Plotly, Bokeh e Altair](./reference_materials/07_visualizacao_interativa_com_plotly_e_bokeh.pdf)
- [Cheat sheet — Interatividade](./cheat_sheets/cheat_sheet_interatividade.pdf)

Os arquivos estão nesta mesma pasta da disciplina. Os PDFs complementam, mas não substituem, a execução das células.

## 1. Plotly Express — gráficos interativos

Plotly oferece zoom, seleção, hover e exportação interativa. A exibição depende do renderer do JupyterLab; algumas visualizações podem não aparecer na prévia estática do GitHub.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import plotly.io as pio

pio.renderers.default = "iframe"
pio.renderers["iframe"].html_directory = "outputs/iframe_figures"

def preparar_pastas():
    """Localiza a biblioteca e cria as pastas de dados e resultados, se ausentes."""
    atual = Path.cwd().resolve()
    # Aceita execução na raiz da biblioteca ou em uma pasta filha.
    candidatos = [atual, atual.parent]
    raiz = next(
        (p for p in candidatos if (p / 'vendas_exemplo.csv').is_file() or
         (p / 'datasets' / 'vendas_exemplo.csv').is_file()),
        None,
    )
    if raiz is None:
        # Identifica a raiz da biblioteca pelo conjunto de notebooks.
        raiz = next(
            (p for p in candidatos if (p / '01_Analise_Exploratoria_de_Dados.ipynb').is_file()),
            atual,
        )
    dados = raiz / 'datasets'
    saidas = raiz / 'outputs'
    dados.mkdir(parents=True, exist_ok=True)
    saidas.mkdir(parents=True, exist_ok=True)
    return dados, saidas

DATA, OUTPUTS = preparar_pastas()

def conferir_datasets(*nomes):
    ausentes = [nome for nome in nomes if not (DATA / nome).is_file()]
    if ausentes:
        raise FileNotFoundError(
            'Datasets ausentes: ' + ', '.join(ausentes) + '\n'
            + 'Coloque os CSVs fornecidos em: ' + str(DATA.resolve()) + '\n'
            + 'Criar a pasta datasets/ não cria automaticamente os arquivos CSV.'
        )
conferir_datasets('vendas_exemplo.csv', 'clima_sintetico.csv')
import plotly.express as px
vendas=pd.read_csv(DATA/'vendas_exemplo.csv').drop_duplicates()
agrupado=vendas.groupby('produto',as_index=False)['unidades'].sum()
fig=px.bar(agrupado,x='produto',y='unidades',title='Unidades por produto (simulação)',labels={'unidades':'Unidades','produto':'Produto'})
fig.update_layout(template='plotly_white')
fig.show()

## 2. Plotly — série temporal com hover

In [2]:
clima=pd.read_csv(DATA/'clima_sintetico.csv',parse_dates=['data'])
fig=px.line(clima,x='data',y='temperatura_c',title='Temperatura simulada',labels={'temperatura_c':'°C','data':'Data'})
fig.update_traces(hovertemplate='%{x|%d/%m/%Y}<br>%{y:.1f} °C<extra></extra>')
fig.show()

## 3. Visualização geográfica — exemplo esquemático

Para mapas reais, utilize coordenadas verificadas e dados com fonte e licença. Este exemplo usa localizações aproximadas para demonstrar a API; não representa indicadores geográficos reais.

In [3]:
pontos=pd.DataFrame({'cidade':['São Paulo','Rio de Janeiro','Belo Horizonte'],'lat':[-23.55,-22.91,-19.92],'lon':[-46.63,-43.20,-43.94],'valor_exemplo':[10,20,15]})
fig=px.scatter_geo(pontos,lat='lat',lon='lon',hover_name='cidade',size='valor_exemplo',scope='south america',title='Pontos demonstrativos (valores fictícios)')
fig.show()

## 4. Bokeh — interatividade em HTML

Bokeh utiliza fontes de dados e ferramentas interativas. Para abrir no navegador, salve HTML em pasta de saída; não há necessidade de servidor para este exemplo.

In [4]:
from bokeh.plotting import figure, show, output_notebook
from bokeh.models import ColumnDataSource, HoverTool
from bokeh.io import output_file, reset_output
from bokeh.resources import CDN
source=ColumnDataSource(data={'produto':agrupado['produto'].tolist(),'unidades':agrupado['unidades'].tolist()})
p=figure(x_range=agrupado['produto'].tolist(),height=300,width=700,title='Vendas sintéticas por produto',tools='pan,wheel_zoom,box_zoom,reset,save')
p.vbar(x='produto',top='unidades',width=.6,source=source)
p.add_tools(HoverTool(tooltips=[('Produto','@produto'),('Unidades','@unidades')]))
p.xaxis.axis_label='Produto';p.yaxis.axis_label='Unidades'
# Exibição opcional em JupyterLab: descomente as duas linhas abaixo.
# output_notebook()
# show(p)
print('Objeto Bokeh criado. Use output_notebook() e show(p) para exibir.')

Objeto Bokeh criado. Use output_notebook() e show(p) para exibir.


## 5. Exercício resolvido — gráfico interativo com agregação

**Enunciado:** calcule o número de unidades por região e represente com hover contendo a contagem.

In [5]:
regiao=vendas.groupby('regiao',dropna=False,as_index=False)['unidades'].sum().fillna({'regiao':'Não informado'})
fig=px.bar(regiao,x='regiao',y='unidades',title='Unidades por região (dados sintéticos)',hover_data={'unidades':':,.0f'})
fig.show()

## 6. Plotly Graph Objects, subplots e apresentação
Plotly Express resolve gráficos comuns; `graph_objects` oferece controle fino. `make_subplots` permite combinar indicadores com escalas separadas. Evite um segundo eixo quando a comparação ficar ambígua.

In [6]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
clim = pd.read_csv(DATA/'clima_sintetico.csv', parse_dates=['data']).sort_values('data')
mensal = clim.set_index('data').resample('MS').agg(temp=('temperatura_c','mean'), chuva=('chuva_mm','sum')).reset_index()
fig = make_subplots(rows=1, cols=2, subplot_titles=('Temperatura média (°C)', 'Chuva acumulada (mm)'))
fig.add_trace(go.Scatter(x=mensal['data'], y=mensal['temp'], mode='lines+markers', name='Temperatura'), row=1, col=1)
fig.add_trace(go.Bar(x=mensal['data'], y=mensal['chuva'], name='Chuva'), row=1, col=2)
fig.update_layout(title='Indicadores mensais simulados', template='plotly_white', showlegend=False, height=390)
fig.show()

## 7. Hover customizado, botões e exportação HTML
A interação deve ajudar a responder uma pergunta. Evite excesso de controles. `write_html` cria um arquivo portátil em `outputs/`, com JavaScript incluído no próprio HTML.

In [7]:
from pathlib import Path
# OUTPUTS foi preparado na primeira célula
reg = vendas.assign(receita=lambda d: d['unidades']*d['preco_unitario']).groupby('regiao', dropna=False, as_index=False)['receita'].sum(min_count=1)
fig = go.Figure(go.Bar(x=reg['regiao'].fillna('Não informada'), y=reg['receita'], customdata=reg[['receita']],
                       hovertemplate='<b>%{x}</b><br>Receita: %{y:,.2f}<extra></extra>'))
fig.update_layout(title='Receita por região — simulação', template='plotly_white', yaxis_title='Receita',
                  updatemenus=[dict(type='buttons', direction='right', buttons=[
                    dict(label='Barras', method='restyle', args=[{'type':'bar'}]),
                    dict(label='Dispersão', method='restyle', args=[{'type':'scatter','mode':'markers'}])])])
fig.write_html(OUTPUTS/'receita_interativa.html', include_plotlyjs=True, full_html=True)
print('HTML salvo em:', (OUTPUTS/'receita_interativa.html').resolve())
fig.show()

HTML salvo em: /run/media/eduardo/Backup/Faculdades/PUC - CIENCIA DE DADOS/PUC/02_Disciplinas/02_data_analysis_visualization/outputs/receita_interativa.html


## 8. Facetas e mapas com Plotly
Para dados geográficos reais, use coordenadas georreferenciadas e fonte explícita. O exemplo de pontos usa posições conhecidas de cidades, mas o indicador exibido continua **fictício**.

In [8]:
fig = px.scatter_geo(pontos, lat='lat', lon='lon', hover_name='cidade', size='valor_exemplo',
                     scope='south america', title='Cidades com métricas fictícias — mapa demonstrativo',
                     projection='natural earth')
fig.update_traces(marker=dict(opacity=.7))
fig.show()
fig = px.histogram(vendas, x='unidades', facet_col='produto', nbins=10, title='Unidades por produto — simulação')
fig.update_layout(template='plotly_white', height=320)
fig.show()

## 9. Bokeh: fonte de dados, hover e exportação
Bokeh mantém uma fonte estruturada (`ColumnDataSource`) que pode alimentar diferentes marcas gráficas. Use `save()` para gerar HTML sem depender da renderização do notebook.

In [9]:
from bokeh.plotting import figure, save
from bokeh.models import ColumnDataSource, HoverTool
from bokeh.resources import INLINE
from bokeh.io import output_file, reset_output
ag = vendas.groupby('produto', as_index=False)['unidades'].sum()
source2 = ColumnDataSource(dict(produto=ag['produto'].astype(str).tolist(), unidades=ag['unidades'].tolist()))
p2 = figure(x_range=ag['produto'].astype(str).tolist(), height=330, width=760,
            title='Unidades por produto (simulação)', tools='pan,wheel_zoom,box_zoom,reset,save')
p2.vbar(x='produto', top='unidades', width=.7, source=source2)
p2.add_tools(HoverTool(tooltips=[('Produto','@produto'),('Unidades','@unidades')]))
p2.xaxis.axis_label='Produto';p2.yaxis.axis_label='Unidades'
reset_output()
output_file(OUTPUTS/'bokeh_produtos.html', title='Gráfico Bokeh')
save(p2, resources=INLINE)
print('HTML Bokeh salvo em:', (OUTPUTS/'bokeh_produtos.html').resolve())

HTML Bokeh salvo em: /run/media/eduardo/Backup/Faculdades/PUC - CIENCIA DE DADOS/PUC/02_Disciplinas/02_data_analysis_visualization/outputs/bokeh_produtos.html


## 10. Bokeh: seleção e atualização por widget no navegador
A interface abaixo usa um `Select` do Bokeh e um callback JavaScript local para trocar a métrica apresentada, sem um servidor Python. O resultado é exportado em HTML para abrir no navegador.

In [10]:
from bokeh.models import Select, CustomJS
from bokeh.layouts import column
res = vendas.assign(receita=lambda d: d['unidades']*d['preco_unitario']).groupby('produto').agg(unidades=('unidades','sum'), receita=('receita', lambda s: s.sum(min_count=1))).reset_index()
fonte = ColumnDataSource(dict(produto=res['produto'].tolist(), unidades=res['unidades'].tolist(), receita=res['receita'].tolist(), exibido=res['unidades'].tolist()))
p3 = figure(x_range=res['produto'].tolist(), height=300, width=700, title='Alterar métrica')
p3.vbar(x='produto', top='exibido', source=fonte, width=.7)
p3.add_tools(HoverTool(tooltips=[('Produto','@produto'),('Valor','@exibido{0,0.00}')]))
seletor = Select(title='Métrica:', value='unidades', options=['unidades','receita'])
seletor.js_on_change('value', CustomJS(args=dict(fonte=fonte), code="""
    const d = fonte.data; const chave = cb_obj.value;
    d.exibido = Array.from(d[chave]); fonte.change.emit();
"""))
reset_output();output_file(OUTPUTS/'bokeh_interativo.html', title='Bokeh interativo')
save(column(seletor, p3), resources=INLINE)
print('Bokeh interativo salvo em:', (OUTPUTS/'bokeh_interativo.html').resolve())

Bokeh interativo salvo em: /run/media/eduardo/Backup/Faculdades/PUC - CIENCIA DE DADOS/PUC/02_Disciplinas/02_data_analysis_visualization/outputs/bokeh_interativo.html


## 11. Exercício resolvido — painel com duas leituras
**Enunciado:** calcule quantidade e receita por produto, componha dois gráficos no mesmo HTML e permita que o leitor compare métricas sem confundir unidades.

In [11]:
painel = make_subplots(rows=1, cols=2, subplot_titles=('Unidades', 'Receita'))
painel.add_trace(go.Bar(x=res['produto'], y=res['unidades'], name='Unidades'),row=1,col=1)
painel.add_trace(go.Bar(x=res['produto'], y=res['receita'], name='Receita'),row=1,col=2)
painel.update_layout(title='Painel comparativo — dados sintéticos', template='plotly_white', showlegend=False)
painel.write_html(OUTPUTS/'painel_comparativo.html', include_plotlyjs=True)
assert (OUTPUTS/'painel_comparativo.html').exists()
print('Painel exportado para outputs/painel_comparativo.html')

Painel exportado para outputs/painel_comparativo.html


### Consulta rápida — interatividade
- `px.bar`, `px.line`, `px.scatter`: entrada rápida com DataFrames.
- `go.Figure`, `make_subplots`: personalização e combinação.
- `fig.write_html(..., include_plotlyjs=True)`: HTML portátil.
- `ColumnDataSource` + `HoverTool`: dados e interação no Bokeh.
- `CustomJS` + widget: interação em HTML sem servidor.
- Gráfico interativo **não substitui** explicação da pergunta, unidades e limites.

## 12. Altair — visualização declarativa

Altair descreve **o que mapear** (campos, tipos e codificações), em vez de desenhar marcas passo a passo. É útil para exploração comparável e gráficos declarativos concisos.

Neste exercício usamos `mark_bar()`, `encode()`, `tooltip` e `interactive()`; a mesma agregação da base permite confrontar o resultado com Plotly e Bokeh. É necessário instalar `altair` no ambiente do kernel (`requirements.txt`). Para visualizar no JupyterLab, execute a célula; para exportar HTML, o gráfico é gravado em `outputs/`.

Uma vantagem é a clareza dos tipos: `:N` é nominal, `:Q` é quantitativo e `:T` é temporal.

In [12]:
import altair as alt

base_altair = vendas.assign(receita=lambda d: d['unidades'] * d['preco_unitario'])
resumo_altair = (base_altair.groupby('produto', as_index=False)
                 .agg(unidades=('unidades', 'sum'),
                      receita_conhecida=('receita', lambda s: s.sum(min_count=1))))
grafico_altair = alt.Chart(resumo_altair).mark_bar().encode(
    x=alt.X('unidades:Q', title='Unidades vendidas'),
    y=alt.Y('produto:N', sort='-x', title='Produto'),
    tooltip=[alt.Tooltip('produto:N', title='Produto'),
             alt.Tooltip('unidades:Q', title='Unidades', format=',.0f')]
).properties(title='Unidades por produto — dados sintéticos', width=550, height=250).interactive()
grafico_altair.save(OUTPUTS/'altair_produtos.html')
grafico_altair
print('Altair: HTML salvo em', (OUTPUTS/'altair_produtos.html').resolve())
assert (OUTPUTS/'altair_produtos.html').is_file()
assert resumo_altair['unidades'].sum() == vendas['unidades'].sum()

Altair: HTML salvo em /run/media/eduardo/Backup/Faculdades/PUC - CIENCIA DE DADOS/PUC/02_Disciplinas/02_data_analysis_visualization/outputs/altair_produtos.html


## 13. Exercício resolvido — comparação de ferramentas

**Enunciado:** a partir do mesmo agrupamento de unidades por produto, escolha a biblioteca mais apropriada para três situações:

1. Comparação interativa rápida com diversos tipos de gráfico.
2. Um painel HTML com widget e atualização JavaScript sem servidor.
3. Uma especificação compacta de marcas, eixos e tooltips baseada em campos.

**Resolução e interpretação:** Plotly Express atende ao caso 1; Bokeh com `ColumnDataSource` e `CustomJS` ao caso 2; Altair com `mark_bar` e `encode` ao caso 3. Não existe vencedor universal. Todos precisam de fontes, unidades e limitações explícitas.

**Verificação:** as três abordagens devem partir das mesmas somas; diferenças de aparência não podem mudar os dados.

In [13]:
controle_unidades = vendas.groupby('produto')['unidades'].sum().sort_index()
assert controle_unidades.equals(resumo_altair.set_index('produto')['unidades'].sort_index())
print('Somas de unidades por produto conferidas:')
print(controle_unidades.to_string())

Somas de unidades por produto conferidas:
produto
Café    412
Chá     551
Suco    644
Água    768


## Consulta rápida

| Necessidade | Plotly | Bokeh | Altair |
|---|---|---|---|
| Gráfico rápido | `px.bar`, `px.line`, `px.scatter` | `figure`, `vbar`, `line`, `circle` | `alt.Chart(...).mark_bar().encode(...)` |
| Hover | `hover_data`, `hovertemplate` | `HoverTool` | `tooltip=[...]` |
| HTML | `fig.write_html(...)` | `output_file(...)`, `save(...)` | `.save("outputs/arquivo.html")` |
| Integração Jupyter | `fig.show()` | `output_notebook()` + `show(...)` | exibir objeto `Chart` |

O código não substitui o estudo de bibliotecas de aplicação web; essa expansão fica reservada para trabalhos posteriores.

**Limitação:** receita conhecida não representa necessariamente a receita real quando há preços ausentes; mapas são ilustrativos.

## Aprofundamento v2 — acessibilidade, exportação e diagnóstico de renderização

**Decisão já validada:** manter `pio.renderers.default = "iframe"` para o seu JupyterLab. O renderer `plotly_mimetype` havia impedido a exibição dos gráficos em uma sessão anterior. O funcionamento depende do ambiente; após instalar componentes de interface, reinicie **servidor e kernel** do JupyterLab antes de concluir que há incompatibilidade.

`iframe_figures/` contém HTML gerado pelo Plotly: não faz parte dos datasets e **não precisa ser distribuído**. Quando a exibição embutida falhar, você pode abrir o HTML exportado em `outputs/`.

**Exercício resolvido:** criar uma exportação leve com o Plotly JS via CDN para inspeção local quando houver internet. Para um HTML totalmente offline, use `include_plotlyjs=True`, que aumenta o arquivo.

In [14]:
from pathlib import Path
import plotly.express as px
import pandas as pd
import plotly.io as pio
assert pio.renderers.default == 'iframe'
base_v2 = pd.read_csv(DATA / 'vendas_exemplo.csv')
serie_v2 = base_v2.groupby('produto',as_index=False)['unidades'].sum()
fig_v2 = px.bar(serie_v2,x='produto',y='unidades',title='Unidades por produto — dados sintéticos',
                labels={'produto':'Produto','unidades':'Unidades observadas'})
destino_v2 = OUTPUTS / 'grafico_v2_cdn.html'
fig_v2.write_html(destino_v2,include_plotlyjs='cdn',full_html=True)
assert destino_v2.is_file() and destino_v2.stat().st_size > 1000
print('HTML de apoio gerado:',destino_v2)
print('Para uso totalmente offline, troque include_plotlyjs por True.')

HTML de apoio gerado: /run/media/eduardo/Backup/Faculdades/PUC - CIENCIA DE DADOS/PUC/02_Disciplinas/02_data_analysis_visualization/outputs/grafico_v2_cdn.html
Para uso totalmente offline, troque include_plotlyjs por True.


**Boas práticas de interatividade:** hover com unidade explícita, contraste adequado, legenda verificável, fonte visível, tamanho de arquivo documentado e alternativa tabular para quem não consegue interagir com o gráfico.